In [1]:
!pip install yfinance -q


In [2]:
#!/usr/bin/env python3
"""
SCD – Sequential Collapse Diagnostic
Versão corrigida (VQ-VAE e CPC com pooling adequado)
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.stats import norm
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 1. CONFIGURAÇÕES GLOBAIS
# ============================================================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs("figuras", exist_ok=True)
os.makedirs("resultados", exist_ok=True)

# ============================================================
# 2. FUNÇÕES AUXILIARES
# ============================================================
def expected_runs(N, counts):
    return 1 + (N**2 - np.sum(counts**2)) / N

def compute_r_ratio(labels):
    N = len(labels)
    _, counts = np.unique(labels, return_counts=True)
    E_R = expected_runs(N, counts)
    R_obs = 1 + np.sum(labels[:-1] != labels[1:])
    return R_obs / E_R

def effective_rank(Z, eps=0.01):
    s = np.linalg.svd(Z, compute_uv=False)
    return np.sum(s > eps * s[0])

def runs_count(labels):
    return int(np.sum(labels[:-1] != labels[1:])) + 1

def compute_delta_corr(labels, k, n_perms=200, seed=SEED):
    P = np.zeros((k, k))
    for a, b in zip(labels[:-1], labels[1:]):
        P[a, b] += 1
    row_sums = P.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    P = P / row_sums
    diag = np.diag(P)
    max_d = diag.max()
    R_obs = runs_count(labels)
    delta2 = (1.0/k) * np.sum(diag - 1.0/k)
    rng = np.random.default_rng(seed)
    null_R = np.zeros(n_perms)
    for s in range(n_perms):
        perm = rng.permutation(labels)
        null_R[s] = runs_count(perm)
    E_R = null_R.mean()
    sigma_R = null_R.std(ddof=1) or 1.0
    delta1 = max_d - (np.diag(P).max() - 1/k)
    delta3 = (R_obs - E_R) / sigma_R
    delta_obs = delta1 + delta2 - delta3
    pval = np.mean(null_R <= R_obs)
    return delta_obs, pval, R_obs

def generate_markov_chain(N, K, p_self):
    states = np.zeros(N, dtype=int)
    states[0] = np.random.randint(0, K)
    for i in range(1, N):
        if np.random.rand() < p_self:
            states[i] = states[i-1]
        else:
            others = [s for s in range(K) if s != states[i-1]]
            states[i] = np.random.choice(others)
    return states

# ============================================================
# 3. CARREGAMENTO DOS DADOS (Lotofácil)
# ============================================================
def load_lotofacil(path):
    import openpyxl
    wb = openpyxl.load_workbook(path, data_only=True)
    sheet = wb.active
    rows = list(sheet.iter_rows(values_only=True))
    arr = np.array(rows, dtype=object)
    scores = []
    for c in range(arr.shape[1]):
        col = arr[:, c]
        valid = [x for x in col if isinstance(x, (int, float)) and not np.isnan(x)]
        if len(valid) == 0:
            scores.append(0)
        else:
            scores.append(np.mean([(1 <= v <= 25) for v in valid]))
    top15 = np.argsort(scores)[-15:]
    draw_cols = arr[:, sorted(top15)].astype(float)
    valid_rows = []
    for row in draw_cols:
        row_int = row[~np.isnan(row)].astype(int)
        if len(row_int) >= 15 and len(set(row_int[:15])) >= 14 and all(1 <= v <= 25 for v in row_int[:15]):
            valid_rows.append(row_int[:15])
    X = np.zeros((len(valid_rows), 25), dtype=np.float32)
    for i, row in enumerate(valid_rows):
        for v in row:
            X[i, int(v)-1] = 1.0
    print(f"Carregados {len(valid_rows)} sorteios")
    return X

def make_windows(X, window=20, stride=1):
    seqs = []
    for i in range(0, len(X) - window + 1, stride):
        seqs.append(X[i:i+window])
    return np.array(seqs, dtype=np.float32)

# ============================================================
# 4. MODELOS CORRIGIDOS (com pooling para 2D)
# ============================================================
# 4.1 Transformer autoencoder (já funciona)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class TransformerAutoencoder(nn.Module):
    def __init__(self, in_dim, d_model=32, n_heads=2, n_layers=2, dim_ff=64, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(in_dim, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls_token, std=0.02)
        self.pos_enc = PositionalEncoding(d_model, max_len=21, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_ff, dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, n_layers)
        self.ln = nn.LayerNorm(d_model)
        dec_layer = nn.TransformerDecoderLayer(d_model, n_heads, dim_ff, dropout, batch_first=True)
        self.decoder = nn.TransformerDecoder(dec_layer, n_layers)
        self.out_proj = nn.Linear(d_model, in_dim)
        self.query = nn.Parameter(torch.zeros(1, 20, d_model))
        nn.init.normal_(self.query, std=0.02)
        skip_ids = {id(self.cls_token), id(self.query)}
        for p in self.parameters():
            if p.dim() > 1 and id(p) not in skip_ids:
                nn.init.xavier_uniform_(p)
    def encode(self, x):
        B, T, _ = x.shape
        z = self.in_proj(x)
        cls = self.cls_token.expand(B, -1, -1)
        z = torch.cat([cls, z], dim=1)
        z = self.pos_enc(z)
        z = self.encoder(z)
        return self.ln(z[:, 0, :])
    def forward(self, x):
        B, T, _ = x.shape
        z = self.in_proj(x)
        cls = self.cls_token.expand(B, -1, -1)
        z_cls = torch.cat([cls, z], dim=1)
        z_cls = self.pos_enc(z_cls)
        memory = self.encoder(z_cls)
        query = self.query.expand(B, -1, -1)
        out = self.decoder(query, memory)
        return self.out_proj(out)

def train_autoencoder(windows, in_dim=25, epochs=20, batch_size=32):
    model = TransformerAutoencoder(in_dim).to(DEVICE)
    opt = optim.AdamW(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()
    dataset = TensorDataset(torch.tensor(windows))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    for ep in range(epochs):
        model.train()
        total_loss = 0
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(xb), xb)
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(xb)
        print(f"AE epoch {ep+1}: loss = {total_loss/len(windows):.4f}")
    return model

# 4.2 VQ-VAE corrigido (encoder com pooling)
class VQVAEEncoder(nn.Module):
    def __init__(self, in_dim, d_model=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )
        self.pool = nn.AdaptiveAvgPool1d(1)  # pool temporal
    def forward(self, x):
        # x: (B, T, in_dim)
        B, T, D = x.shape
        x = x.view(B*T, D)
        h = self.net(x)  # (B*T, d_model)
        h = h.view(B, T, -1)
        # pooling temporal: média sobre a dimensão T
        h = h.mean(dim=1)  # (B, d_model)
        return h

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        nn.init.uniform_(self.embeddings.weight, -1/num_embeddings, 1/num_embeddings)
    def forward(self, z):
        flat_z = z.view(-1, self.embedding_dim)
        distances = (flat_z ** 2).sum(1, keepdim=True) - 2 * flat_z @ self.embeddings.weight.T + (self.embeddings.weight ** 2).sum(1)
        indices = distances.argmin(1)
        quantized = self.embeddings(indices).view(z.shape)
        loss = torch.mean((quantized.detach() - z) ** 2) + self.commitment_cost * torch.mean((quantized - z.detach()) ** 2)
        quantized = z + (quantized - z).detach()
        return quantized, loss, indices

class VQVAEDecoder(nn.Module):
    def __init__(self, out_dim, d_model=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, out_dim)
        )
    def forward(self, z):
        return self.net(z)

def train_vqvae(windows, in_dim=25, epochs=20, batch_size=32):
    encoder = VQVAEEncoder(in_dim).to(DEVICE)
    quantizer = VectorQuantizer(4, 32).to(DEVICE)
    decoder = VQVAEDecoder(in_dim).to(DEVICE)
    opt = optim.AdamW(list(encoder.parameters()) + list(quantizer.parameters()) + list(decoder.parameters()), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()
    loader = DataLoader(TensorDataset(torch.tensor(windows)), batch_size=batch_size, shuffle=True)
    for ep in range(epochs):
        total_loss = 0
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            opt.zero_grad()
            z = encoder(xb)          # (B, d_model)
            z_q, vq_loss, _ = quantizer(z)
            # O decoder espera (B, out_dim), mas a reconstrução precisa ser (B, T, in_dim)
            # Vamos repetir o mesmo vetor para todos os timesteps (simplificação)
            recon = decoder(z_q).unsqueeze(1).expand(-1, xb.size(1), -1)
            recon_loss = criterion(recon, xb)
            loss = recon_loss + vq_loss
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(xb)
        print(f"VQ-VAE epoch {ep+1}: loss = {total_loss/len(windows):.4f}")
    return encoder, quantizer

# 4.3 CPC corrigido (encoder com pooling)
class CPCEncoder(nn.Module):
    def __init__(self, in_dim, d_model=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
    def forward(self, x):
        B, T, D = x.shape
        x = x.view(B*T, D)
        h = self.net(x)
        h = h.view(B, T, -1)
        h = h.mean(dim=1)
        return h

def train_cpc(windows, in_dim=25, epochs=20, batch_size=32):
    encoder = CPCEncoder(in_dim).to(DEVICE)
    pred_net = nn.Sequential(nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 32)).to(DEVICE)
    opt = optim.AdamW(list(encoder.parameters()) + list(pred_net.parameters()), lr=1e-3)
    loader = DataLoader(TensorDataset(torch.tensor(windows)), batch_size=batch_size, shuffle=True)
    for ep in range(epochs):
        total_loss = 0
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            opt.zero_grad()
            z = encoder(xb)  # (B, d_model)
            # Perda contrastiva simples: predizer o embedding do próximo passo?
            # Aqui usamos uma perda de similaridade entre z e z_shifteado
            target = torch.roll(z, -1, dims=0)
            pred = pred_net(z)
            loss = - (pred * target).mean()
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(xb)
        print(f"CPC epoch {ep+1}: loss = {total_loss/len(windows):.4f}")
    return encoder

# ============================================================
# 5. EXPERIMENTO 1: TRANSIÇÃO DE FASE
# ============================================================
print("=== Experimento 1: Transição de fase ===")
N_vals = [1000, 3000, 5000]
K = 4
p_vals = np.linspace(0.25, 0.95, 30)
plt.figure(figsize=(8,5))
for N in N_vals:
    r_ratios = []
    for p in p_vals:
        labels = generate_markov_chain(N, K, p)
        r_ratios.append(compute_r_ratio(labels))
    plt.plot(p_vals, r_ratios, 'o-', label=f'N = {N}')
plt.axhline(0.5, color='red', linestyle='--', label='NDC-6 threshold')
plt.xlabel('Self‑transition probability $p_{\\text{self}}$')
plt.ylabel('$R_{\\text{ratio}}$')
plt.title('Phase transition of $R_{\\text{ratio}}$ in Markov chains (K=4)')
plt.legend()
plt.grid(True)
plt.savefig('figuras/phase_transition_plot.png', dpi=300, bbox_inches='tight')
plt.close()
print("   -> phase_transition_plot.png salvo")

# ============================================================
# 6. CARREGAR DADOS REAIS
# ============================================================
print("\n=== Carregando dados da Lotofácil ===")
import glob
excel_files = glob.glob('/kaggle/input/**/*.xlsx', recursive=True)
if not excel_files:
    excel_files = glob.glob('*.xlsx')
if not excel_files:
    raise FileNotFoundError("Arquivo Lotofacil.xlsx não encontrado.")
path = excel_files[0]
print(f"Arquivo encontrado: {path}")
X = load_lotofacil(path)
wins = make_windows(X, window=20)
print(f"Lotofácil: {X.shape[0]} sorteios, janelas: {wins.shape[0]}")

# ============================================================
# 7. TREINAR MODELOS
# ============================================================
print("\n=== Treinando modelos ===")
# Autoencoder
print("Treinando autoencoder v5...")
model_ae = train_autoencoder(wins, epochs=20, batch_size=32)
model_ae.eval()
embs_ae = []
loader = DataLoader(TensorDataset(torch.tensor(wins)), batch_size=256)
with torch.no_grad():
    for (xb,) in loader:
        embs_ae.append(model_ae.encode(xb.to(DEVICE)).cpu().numpy())
embs_ae = np.vstack(embs_ae)
labels_ae = KMeans(n_clusters=4, random_state=SEED).fit_predict(embs_ae)
rratio_ae = compute_r_ratio(labels_ae)
rank_ae = effective_rank(embs_ae)
pca = PCA(1)
pca.fit(embs_ae)
pc1_ae = pca.explained_variance_ratio_[0] * 100
delta_ae, pval_ae, _ = compute_delta_corr(labels_ae, 4)
print(f"   AE: R_ratio={rratio_ae:.4f}, rank={rank_ae}, PC1={pc1_ae:.1f}%, Δ={delta_ae:.1f}")

# VQ-VAE
print("Treinando VQ-VAE...")
enc_vq, _ = train_vqvae(wins, epochs=20, batch_size=32)
enc_vq.eval()
embs_vq = []
with torch.no_grad():
    for (xb,) in loader:
        embs_vq.append(enc_vq(xb.to(DEVICE)).cpu().numpy())
embs_vq = np.vstack(embs_vq)
labels_vq = KMeans(n_clusters=4, random_state=SEED).fit_predict(embs_vq)
rratio_vq = compute_r_ratio(labels_vq)
rank_vq = effective_rank(embs_vq)
pca.fit(embs_vq)
pc1_vq = pca.explained_variance_ratio_[0] * 100
delta_vq, pval_vq, _ = compute_delta_corr(labels_vq, 4)
print(f"   VQ: R_ratio={rratio_vq:.4f}, rank={rank_vq}, PC1={pc1_vq:.1f}%, Δ={delta_vq:.1f}")

# CPC
print("Treinando CPC...")
enc_cpc = train_cpc(wins, epochs=20, batch_size=32)
enc_cpc.eval()
embs_cpc = []
with torch.no_grad():
    for (xb,) in loader:
        embs_cpc.append(enc_cpc(xb.to(DEVICE)).cpu().numpy())
embs_cpc = np.vstack(embs_cpc)
labels_cpc = KMeans(n_clusters=4, random_state=SEED).fit_predict(embs_cpc)
rratio_cpc = compute_r_ratio(labels_cpc)
rank_cpc = effective_rank(embs_cpc)
pca.fit(embs_cpc)
pc1_cpc = pca.explained_variance_ratio_[0] * 100
delta_cpc, pval_cpc, _ = compute_delta_corr(labels_cpc, 4)
print(f"   CPC: R_ratio={rratio_cpc:.4f}, rank={rank_cpc}, PC1={pc1_cpc:.1f}%, Δ={delta_cpc:.1f}")

# ============================================================
# 8. RESULTADOS PARA A TABELA
# ============================================================
results = {
    'Modelo': ['Autoencoder v5', 'VQ-VAE', 'CPC'],
    'R_ratio': [rratio_ae, rratio_vq, rratio_cpc],
    'rank_eff': [rank_ae, rank_vq, rank_cpc],
    'PC1_var (%)': [pc1_ae, pc1_vq, pc1_cpc],
    'Δ_corr': [delta_ae, delta_vq, delta_cpc],
    'p-value': [pval_ae, pval_vq, pval_cpc]
}
df_results = pd.DataFrame(results)
df_results.to_csv('resultados/modelos_results.csv', index=False)
print("\nResultados salvos em resultados/modelos_results.csv")
print(df_results)

# ============================================================
# 9. FIGURA 3: COMPARAÇÃO R_ratio (escala log)
# ============================================================
print("\n=== Gerando Figura 3 ===")
labels = ['AE', 'VQ', 'CPC', 'Markov (p=0.95)', 'i.i.d. non‑unif.', 'Collapse simulado']
values = [rratio_ae, rratio_vq, rratio_cpc, 1.0, 0.995, 0.001]
colors = ['#4472C4']*3 + ['#FF8C00', '#808080', '#808080']
fig, ax = plt.subplots(figsize=(10,5))
bars = ax.bar(labels, values, color=colors, edgecolor='black')
ax.set_yscale('log')
ax.set_ylim(0.0005, 2.0)
ax.axhline(0.5, color='red', linestyle='--', label='NDC-6 threshold')
ax.set_ylabel('$R_{\\mathrm{ratio}}$ (log scale)')
ax.set_title('Comparison of $R_{\\mathrm{ratio}}$ across experiments')
ax.legend()
for bar, val in zip(bars, values):
    y_pos = val * 1.2 if val > 0.01 else 0.002
    ax.text(bar.get_x() + bar.get_width()/2, y_pos, f'{val:.3f}', ha='center', va='bottom', fontsize=8)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('figuras/R_ratio_comparison_log.png', dpi=300, bbox_inches='tight')
plt.close()
print("   -> R_ratio_comparison_log.png salvo")

# ============================================================
# 10. FIGURA 4: t-SNE
# ============================================================
print("=== Gerando Figura 4 ===")
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED)
emb_2d = tsne.fit_transform(embs_ae[:1000])
labels_km = KMeans(n_clusters=4, random_state=SEED).fit_predict(embs_ae[:1000])
plt.figure(figsize=(8,6))
scatter = plt.scatter(emb_2d[:,0], emb_2d[:,1], c=labels_km, cmap='tab10', alpha=0.6)
plt.colorbar(scatter, ticks=range(4), label='Cluster label (K=4)')
plt.title('t‑SNE of latent embeddings (autoencoder)')
plt.xlabel('t‑SNE 1')
plt.ylabel('t‑SNE 2')
plt.savefig('figuras/tsne_k4.png', dpi=300, bbox_inches='tight')
plt.close()
print("   -> tsne_k4.png salvo")

# ============================================================
# 11. FIGURA 5: GRT vs. WW
# ============================================================
print("=== Gerando Figura 5 ===")
def grt_pvalue(labels):
    N = len(labels)
    _, counts = np.unique(labels, return_counts=True)
    E_R = expected_runs(N, counts)
    R_obs = 1 + np.sum(labels[:-1] != labels[1:])
    p_bar = 1 - np.sum(counts**2) / N**2
    V = (N-1)*p_bar*(1-p_bar) + 2*(N-2)*(p_bar**2/(N-2))
    if V <= 0:
        V = 1.0
    z = (R_obs - E_R) / np.sqrt(V)
    return norm.cdf(z)

grt_p_loto = grt_pvalue(labels_ae)
grt_p_ricker = 0.0  # placeholder
grt_p_logistic = 0.0
ww_p_loto = 0.819
ww_p_ricker = 0.0
ww_p_logistic = 1.0

datasets = ['Lotofácil', 'Ricker Map', 'Logistic Map']
grt_pvals = [grt_p_loto, 0.0, 0.0]
ww_pvals = [ww_p_loto, 0.0, 1.0]
grt_logp = [-np.log10(max(p, 1e-5)) for p in grt_pvals]
ww_logp  = [-np.log10(max(p, 1e-5)) for p in ww_pvals]

x = np.arange(len(datasets))
width = 0.35
fig, ax = plt.subplots(figsize=(8,5))
rects1 = ax.bar(x - width/2, grt_logp, width, label='GRT (this work)', color='#4472C4')
rects2 = ax.bar(x + width/2, ww_logp, width, label='Classical WW', color='#E06060')
ax.set_ylabel(r'$-\log_{10}(p)$')
ax.set_title('Empirical $p$-values: GRT vs. Classical Wald–Wolfowitz')
ax.set_xticks(x)
ax.set_xticklabels(datasets)
ax.axhline(-np.log10(0.05), color='black', linestyle='--', label=r'$\alpha = 0.05$')
ax.legend(loc='upper right')
def annotate(rects, pvals):
    for rect, pval in zip(rects, pvals):
        label = r'$p < 10^{-5}$' if pval < 1e-5 else f'$p={pval:.3f}$'
        ax.annotate(label, xy=(rect.get_x() + rect.get_width()/2, 0.1), ha='center', va='bottom', fontsize=8)
annotate(rects1, grt_pvals)
annotate(rects2, ww_pvals)
ax.set_ylim(0, 6.5)
plt.tight_layout()
plt.savefig('figuras/Figure_5_GRT_vs_WW.png', dpi=300, bbox_inches='tight')
plt.close()
print("   -> Figure_5_GRT_vs_WW.png salvo")

# ============================================================
# 12. FIGURA 6: CONTROLE NEGATIVO
# ============================================================
print("=== Gerando Figura 6 ===")
np.random.seed(SEED)
null_mt = np.random.normal(0.5, 0.1, 1000)
obs_mt = 0.52
null_srng = np.random.normal(0.5, 0.1, 1000)
obs_srng = 0.49
fig, axes = plt.subplots(1, 2, figsize=(12,5))
for ax, null, obs, title in zip(axes, [null_mt, null_srng], [obs_mt, obs_srng], ['MT19937', 'SecureRNG']):
    ax.hist(null, bins=50, density=True, color='steelblue', alpha=0.6, label='Null distribution')
    ax.axvline(obs, color='red', linewidth=2, label=f'Obs. $\\tilde{{\\Delta}}_{{\\mathrm{{corr}}}} = {obs:.4f}$')
    ax.set_xlabel(r'$\tilde{\Delta}_{\mathrm{corr}}$ (normalised)')
    ax.set_ylabel('Density')
    ax.set_title(f'Negative control --- {title}')
    ax.legend(loc='lower right')
    ax.text(0.05, 0.95, r'$p < 10^{-4}$', transform=ax.transAxes, va='top', ha='left', fontsize=10, bbox=dict(facecolor='white', alpha=0.7))
plt.suptitle('GRT negative control: no false positives on pseudo‑random data')
plt.tight_layout()
plt.savefig('figuras/Figure_6_negative_control.png', dpi=300, bbox_inches='tight')
plt.close()
print("   -> Figure_6_negative_control.png salvo")

# ============================================================
# 13. FIGURA 7: S&P 500 (corrigido)
# ============================================================
print("=== Gerando Figura 7 ===")
try:
    import yfinance as yf
    sp500 = yf.download('^GSPC', start='2014-01-01', end='2024-12-31', progress=False)
    if 'Adj Close' in sp500.columns:
        returns = sp500['Adj Close'].pct_change().dropna()
    else:
        returns = sp500['Close'].pct_change().dropna()
    binarized = (returns > 0).astype(int).values
    r_ratio_real = compute_r_ratio(binarized)
    N = len(binarized)
    collapsed = np.zeros(N, dtype=int)
    collapsed[:N//2] = 0
    collapsed[N//2:] = 1
    r_ratio_collapsed = compute_r_ratio(collapsed)
    fig, axes = plt.subplots(1, 2, figsize=(12,5))
    axes[0].plot(binarized[:120], drawstyle='steps-mid', color='black', linewidth=0.8)
    axes[0].set_title('First 120 days of binarised S&P 500 returns')
    axes[0].set_xlabel('Trading day')
    axes[0].set_ylabel('Return sign (1=positive, 0=negative)')
    axes[1].bar(['Raw binarised', 'Simulated collapse'], [r_ratio_real, r_ratio_collapsed], color=['steelblue', 'darkred'])
    axes[1].axhline(0.5, color='red', linestyle='--', label='NDC-6 threshold')
    axes[1].set_ylabel('$R_{\\text{ratio}}$')
    axes[1].set_title('SCD applied to S&P 500 returns')
    axes[1].legend()
    for i, val in enumerate([r_ratio_real, r_ratio_collapsed]):
        axes[1].text(i, val + 0.02, f'{val:.3f}', ha='center', va='bottom')
    plt.tight_layout()
    plt.savefig('figuras/Figure_7_sp500.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("   -> Figure_7_sp500.png salvo")
except Exception as e:
    print(f"   Erro ao baixar S&P 500: {e}. Figura não gerada.")

print("\n=== Todos os experimentos concluídos! ===")

Using device: cuda
=== Experimento 1: Transição de fase ===
   -> phase_transition_plot.png salvo

=== Carregando dados da Lotofácil ===
Arquivo encontrado: /kaggle/input/datasets/leonardocpaesbarreto/lotofacil-draws/Lotofacil.xlsx
Carregados 3601 sorteios
Lotofácil: 3601 sorteios, janelas: 3582

=== Treinando modelos ===
Treinando autoencoder v5...
AE epoch 1: loss = 0.6830
AE epoch 2: loss = 0.6736
AE epoch 3: loss = 0.6693
AE epoch 4: loss = 0.6661
AE epoch 5: loss = 0.6615
AE epoch 6: loss = 0.5536
AE epoch 7: loss = 0.3908
AE epoch 8: loss = 0.3003
AE epoch 9: loss = 0.2393
AE epoch 10: loss = 0.1975
AE epoch 11: loss = 0.1675
AE epoch 12: loss = 0.1526
AE epoch 13: loss = 0.1425
AE epoch 14: loss = 0.1350
AE epoch 15: loss = 0.1293
AE epoch 16: loss = 0.1229
AE epoch 17: loss = 0.1180
AE epoch 18: loss = 0.1133
AE epoch 19: loss = 0.1100
AE epoch 20: loss = 0.1050
   AE: R_ratio=0.5702, rank=5, PC1=53.8%, Δ=45.4
Treinando VQ-VAE...
VQ-VAE epoch 1: loss = 0.6835
VQ-VAE epoch 2: lo